# Hyperion: training an adverse-selection classifier on simulated order-book features

This notebook trains a small **LightGBM** classifier (10 trees, depth 3) to flag
order-book states that look like informed, one-sided flow — the ticks a market
maker should widen its quotes on.

**Read this first**
* **The features and labels are simulated** (numpy, seed 42), not computed from
  market data. The label is a noisy formula of the *same tick's* features, so a
  high AUC means the model recovered that formula, not that it can predict a
  market. See the [model card](../hf_publish/README.md).
* Step 2 downloads a day of real Binance trades to show where real data would
  come from, but nothing below uses it: trades alone don't carry the order-book
  depth these features need.
* The Rust engine does **not** load the exported model. Its rule
  (`src/quant/ml_model.rs`) is three decision stumps with hand-set thresholds.
* Binance Vision data is licensed CC BY-NC-SA 4.0: research and other
  non-commercial use only.

### Where to run it
* **Google Colab** or **Kaggle Notebooks** (free CPU is plenty)
* Any machine with Python 3.11+

In [ ]:
# 1. Environment Setup
!pip install -q polars lightgbm numpy requests scipy scikit-learn

In [ ]:
# 2. Where real data would come from: one day of Binance trades (not used below)
import io, os, requests, zipfile
import polars as pl
import numpy as np

symbol = "BTCUSDT"
date = "2025-01-15"
url = f"https://data.binance.vision/data/spot/daily/trades/{symbol}/{symbol}-trades-{date}.zip"
print(f"[*] Fetching one day of Binance trades (CC BY-NC-SA 4.0, non-commercial use): {url}")

r = requests.get(url)
if r.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall("data/")
    print("[+] Downloaded and extracted. (Not used below: trades carry no order-book depth.)")
else:
    print(f"[-] Not in the archive (HTTP {r.status_code}). Nothing below depends on it.")

In [ ]:
# 3. Simulated order-book features and a simulated label
print("[*] Simulating 8 order-book features: spread, micro-price skew, depth imbalances, OFI, returns, volatility, trade flow")
np.random.seed(42)
n_samples = 50_000

spread_bps = np.random.exponential(scale=1.2, size=n_samples) + 0.4
imbalance_l0 = np.random.uniform(-1.0, 1.0, size=n_samples)
imbalance_l1 = 0.65 * imbalance_l0 + 0.35 * np.random.uniform(-1.0, 1.0, size=n_samples)
micro_price_bias_bps = 0.85 * imbalance_l0 * spread_bps
ofi = 45.0 * imbalance_l0 + np.random.normal(0, 12.0, size=n_samples)
returns = np.random.normal(0, 0.00015, size=n_samples)
volatility = np.abs(returns) * 120.0
trade_imbalance = np.sign(ofi) * np.random.exponential(scale=2.5, size=n_samples)

X = np.column_stack([
    spread_bps, micro_price_bias_bps, imbalance_l0, imbalance_l1,
    ofi, returns, volatility, trade_imbalance
])

# The label is a noisy threshold on this tick's own features: a model can learn
# it perfectly and still know nothing about what a market does next.
latent_signal = (
    0.35 * micro_price_bias_bps +
    0.025 * ofi +
    0.28 * imbalance_l0 +
    0.15 * trade_imbalance +
    np.random.normal(0, 0.4, size=n_samples)
)
y = np.where(latent_signal > 0.25, 1, 0)
print(f"[+] Constructed feature matrix: {X.shape} with positive class ratio: {np.mean(y)*100:.1f}%")

In [ ]:
# 4. Train LightGBM and score it on the held-out 20%
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, accuracy_score

split = int(0.8 * n_samples)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

train_data = lgb.Dataset(X_train, label=y_train)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'max_depth': 3,
    'num_leaves': 7,
    'learning_rate': 0.1,
    'verbose': -1
}

model = lgb.train(params, train_data, num_boost_round=10)
preds = model.predict(X_test)
auc = roc_auc_score(y_test, preds)
acc = accuracy_score(y_test, (preds >= 0.5).astype(int))

print(f"[+] Held-out AUC: {auc:.4f} | accuracy: {acc*100:.2f}%  (agreement with the simulated label, not market accuracy)")

In [ ]:
# 5. Save the trained trees (LightGBM's JSON dump), for inspection
import json
tree_dict = model.dump_model()

with open("lob_model_weights.json", "w") as f:
    json.dump(tree_dict, f, indent=2)

print("[+] Model exported to lob_model_weights.json!")
print("[*] Note: the Rust engine doesn't read this file. Its rule is hand-set in src/quant/ml_model.rs.")